# Number Image Detection (MNIST)

This notebook introduces neural network training for handwritten digit classification using the MNIST dataset.

**Dataset:** MNIST contains 60,000 training and 10,000 test images of handwritten digits (0-9).

## 1. Load MNIST from Hugging Face

In [2]:
from datasets import load_dataset

ds = load_dataset("ylecun/mnist")
train = ds['train']
test = ds['test']

print(f"Train: {len(train)} samples")
print(f"Test: {len(test)} samples")

/home/pedro/src/ai-notebooks/number-image-detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 60000 samples
Test: 10000 samples


## 2. Simple Perceptron

### Forward Pass
For input $x$, weights $w$, bias $b$, and activation $\sigma$:

$$z = w^T x + b$$
$$\hat{y} = \sigma(z)$$

### Error Functions

**ReLU:** $f(x) = \max(0, x)$

**Sigmoid:** $\sigma(x) = \frac{1}{1 + e^{-x}}$

In [15]:
import numpy as np

# Simple perceptron: single neuron, 784 inputs (flattened 28x28 image)
np.random.seed(42)

# Random weights and bias
w = np.random.randn(784) * 0.01  # small random values
b = np.random.randn() * 0.01

def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Take one sample
image = train[0]['image']
x = np.array(image).flatten().astype(np.float32) / 255.0

# Forward pass: ReLU
z = w @ x + b
a_relu = relu(z)

# Forward pass: Sigmoid
a_sigmoid = sigmoid(z)

print(f"Input shape: {x.shape}")
print(f"z (pre-activation): {z:.4f}")
print(f"ReLU output: {a_relu:.4f}")
print(f"Sigmoid output: {a_sigmoid:.4f}")

Input shape: (784,)
z (pre-activation): 0.1339
ReLU output: 0.1339
Sigmoid output: 0.5334


## 3. Gradient Descent

For a single sample with target $y$ and prediction $\hat{y}$, using MSE loss:

$$L = (y - \hat{y})^2$$

Gradient with respect to weights:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w}$$

$$= -2(y - \hat{y}) \cdot \sigma'(z) \cdot x$$

Update rule:

$$w = w - \alpha \cdot \frac{\partial L}{\partial w}$$

Where $\alpha$ is the learning rate.

In [17]:
def sigmoid_prime(z):
    """Derivative of sigmoid: d/dz[sigmoid(z)] = s(1-s)"""
    s = sigmoid(z)
    return s * (1 - s)


def gradient_descent_step(w, b, x, y_target, alpha):
    """
    One step of gradient descent for a single neuron.
    
    Args:
        w: weights (784,)
        b: bias (scalar)
        x: input image (784,)
        y_target: target value (scalar)
        alpha: learning rate
    
    Returns:
        updated w, updated b, prediction before update
    """
    # Forward pass: compute prediction
    z = w @ x + b          # linear combination
    a = sigmoid(z)         # activation
    
    # Compute error (MSE derivative: dL/dŷ = 2(ŷ - y))
    error = a - y_target
    
    # Gradient of MSE loss w.r.t. prediction
    dL_dy = 2 * error
    
    # Gradient of activation w.r.t. z
    dy_dz = sigmoid_prime(z)
    
    # Chain rule: gradient of loss w.r.t. z
    dL_dz = dL_dy * dy_dz
    
    # Chain rule: gradient of loss w.r.t. weights
    # dz/dw = x, so dL_dw = dL_dz * x
    dL_dw = dL_dz * x
    
    # Update weights and bias
    w = w - alpha * dL_dw
    b = b - alpha * dL_dz
    
    return w, b, a


# Setup
y_true = train[0]['label']
y_target = 1.0 if y_true == 5 else 0.0  # one-hot for digit 5

alpha = 0.1
w_before = w.copy()

# Run 3 gradient descent steps
for step in range(3):
    w, b, pred = gradient_descent_step(w, b, x, y_target, alpha)
    print(f"Step {step+1}: prediction={pred:.4f}, target={y_target}")

# Show weights that changed the most
change = np.abs(w - w_before)
top_indices = np.argsort(change)[-10:][::-1]

print(f"\nTop 10 weight changes after 3 steps:")
for i in top_indices:
    print(f"  w[{i}]: {w_before[i]:.6f} -> {w[i]:.6f} (change: {change[i]:.6f})")

Step 1: prediction=0.9266, target=1.0
Step 2: prediction=0.9326, target=1.0
Step 3: prediction=0.9374, target=1.0

Top 10 weight changes after 3 steps:
  w[161]: 0.033879 -> 0.036458 (change: 0.002580)
  w[234]: 0.047243 -> 0.049803 (change: 0.002559)
  w[651]: 0.037070 -> 0.039629 (change: 0.002559)
  w[494]: 0.041035 -> 0.043595 (change: 0.002559)
  w[602]: 0.034500 -> 0.037059 (change: 0.002559)
  w[521]: 0.031238 -> 0.033797 (change: 0.002559)
  w[600]: 0.033374 -> 0.035933 (change: 0.002559)
  w[574]: 0.034098 -> 0.036657 (change: 0.002559)
  w[654]: 0.051538 -> 0.054097 (change: 0.002559)
  w[235]: 0.032143 -> 0.034703 (change: 0.002559)
